In [4]:
import re
import polars as pl
import plotly.express as px

In [17]:
def parse_perf_state(line):

    if line.startswith("running wasm ..."):
        return { "run": True }

    # Extract timestamp (everything before the colon)
    timestamp_match = re.match(r'^(\d+):', line)
    if not timestamp_match:
        return None
    
    timestamp = int(timestamp_match.group(1))
    
    # Extract the content inside the braces
    braces_match = re.search(r'PerfState\s*{\s*(.*?)\s*}', line, re.DOTALL)
    if not braces_match:
        return None
    
    content = braces_match.group(1)
    
    # Parse key-value pairs
    result = {'timestamp': timestamp}
    
    # Pattern to match key: Some(value)
    pattern = r'(\w+):\s*Some\(([^)]+)\)'
    matches = re.findall(pattern, content)
    
    for key, value in matches:
        # Convert value to appropriate type (int or string)
        try:
            # Try to convert to integer
            result[key] = int(value)
        except ValueError:
            # If conversion fails, keep as string
            result[key] = value
    
    # Set default None values for missing keys
    required_keys = ['cycles', 'cpu_cycles', 'inst_retired', 'l1d_cache', 'l2d_cache']
    for key in required_keys:
        if key not in result:
            result[key] = None
    
    return result


In [26]:
with open("foo.txt") as f:
    lines = f.readlines()

states = []
for line in lines:
    state = parse_perf_state(line)
    if state is not None:
        states.append(state)

states = pl.from_dicts(states)

states = states.with_columns(timestamp = pl.col("timestamp")/1000000, ipc = (pl.col("inst_retired")/pl.col("cpu_cycles")))
states

timestamp,cycles,cpu_cycles,inst_retired,l1d_cache,l2d_cache,ipc
f64,i64,i64,i64,i64,i64,f64
93.223465,14952434,14952445,14493489,5898035,2215,0.969306
93.251217,14952266,14952277,14492454,5897677,2358,0.969247
93.278967,14950119,14950130,14493483,5898034,1801,0.969455
93.30671,14950001,14950000,14493060,5897850,1916,0.969435
93.334454,14950310,14950321,14492877,5897813,1929,0.969402
…,…,…,…,…,…,…
235.365166,14946142,14946153,14613148,5879135,2542,0.97772
235.392999,14947596,14947615,14614095,5879436,2520,0.977687
235.420832,14949012,14949023,14612727,5879016,2615,0.977504


In [34]:
px.line(states, x="timestamp", y=["ipc"])